## 1. Установка инструментов

### 1.1 Базовые библиотеки + эмбеддинги + vector stores

In [ ]:
!pip -q install -U \
  "langchain>=0.2.0" \
  "langchain-community>=0.2.0" \
  "sentence-transformers>=2.6.0" \
  "chromadb>=0.5.0" \
  "faiss-cpu>=1.8.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/1

In [ ]:
import os, sys, subprocess, textwrap

## Один набор эмбеддингов для всех векторных БД

Используем sentence-transformers/all-MiniLM-L6-v2 (быстро и стабильно для демо).

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipython-input-3409896792.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.war

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Данные для демо (документы + разбиение на чанки)

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader

In [ ]:
from typing import List, Dict, Any
from pathlib import Path

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

demo_path = DATA_DIR / "demo_policy.md"
if not demo_path.exists():
    demo_path.write_text(
        "# Политика отпусков (пример)\n"
        "Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.\n"
        "Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.\n"
        "Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.\n",
        encoding="utf-8"
    )

print("Файлы в ./data:")
for p in sorted(DATA_DIR.glob("*")):
    print(" -", p.name)

Файлы в ./data:
 - demo_policy.md


In [ ]:
def load_documents(data_dir: Path) -> List[Document]:
    docs: List[Document] = []
    for path in data_dir.glob("*"):
        suf = path.suffix.lower()
        if suf in [".txt", ".md"]:
            docs.extend(TextLoader(str(path), encoding="utf-8").load())
        elif suf == ".pdf":
            docs.extend(PyPDFLoader(str(path)).load())
    return docs

In [ ]:
docs = load_documents(DATA_DIR)
print(f"Загружено документов: {len(docs)}")
print("Пример:", docs[0].metadata)
print(docs[0].page_content[:400])

Загружено документов: 1
Пример: {'source': 'data/demo_policy.md'}
# Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.



In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
)

chunks = text_splitter.split_documents(docs)
print(f"Чанков: {len(chunks)}")
print("Пример чанка:\n", chunks[0].page_content[:400])

Чанков: 1
Пример чанка:
 # Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.


## Поднимаем vector store (Chroma, FAISS)

### Chroma (in-memory)

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
vs_chroma = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="demo_chroma_inmem",
    persist_directory="./chroma_contracts"
)

In [ ]:
# сохраняем на диск
vs_chroma.persist()

/tmp/ipython-input-2799731612.py:2: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vs_chroma.persist()


In [ ]:
vs_chroma.similarity_search("отпуск", k=5)

[Document(metadata={'source': 'data/demo_policy.md'}, page_content='# Политика отпусков (пример)\nКомпания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.\nСотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.\nДля оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.')]

Переносим на другое хранилище

In [ ]:
store = Chroma(
    collection_name="demo_chroma_inmem",    # Важно! должно совпадать
    embedding_function=embeddings,
    persist_directory="./chroma_contracts"
)

In [ ]:
store.similarity_search("отпуск", k=5)

[Document(metadata={'source': 'data/demo_policy.md'}, page_content='# Политика отпусков (пример)\nКомпания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.\nСотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.\nДля оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.')]

### FAISS (in-memory)

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vs_faiss = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
# сохранить на диск
vs_faiss.save_local("./faiss_index_contracts")

In [ ]:
vs_faiss.similarity_search("отпуск", k=5)

[Document(id='1126c604-fd36-4a5b-91b8-f1eae895ac7b', metadata={'source': 'data/demo_policy.md'}, page_content='# Политика отпусков (пример)\nКомпания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.\nСотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.\nДля оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.')]

Переносим на другое хранилище

In [ ]:
store = FAISS.load_local(
    "./faiss_index_contracts",
    embeddings,
    allow_dangerous_deserialization=True  # потому что pickle
)

store.similarity_search("отпуск", k=5)

[Document(id='1126c604-fd36-4a5b-91b8-f1eae895ac7b', metadata={'source': 'data/demo_policy.md'}, page_content='# Политика отпусков (пример)\nКомпания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.\nСотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.\nДля оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.')]